# Pandas Avanzado — Versión Extendida
Incluye más ejemplos y casos prácticos.

## 1. Indexado avanzado y MultiIndex

In [1]:
import pandas as pd, numpy as np

df = pd.DataFrame({
    'Ciudad':['Madrid','Sevilla','Madrid','Valencia'],
    'Ventas':[100,200,150,130],
    'Unidades':[5,7,6,4]
})
df

,Ciudad,Ventas,Unidades
0,Madrid,100,5
1,Sevilla,200,7
2,Madrid,150,6
3,Valencia,130,4


### Explicación
Creamos un DataFrame base para ejemplificar operaciones avanzadas: servirá para indexado por etiqueta, posición y construcción de MultiIndex.


In [3]:
df.loc[df['Ciudad']=='Madrid']

,Ciudad,Ventas,Unidades
0,Madrid,100,5
2,Madrid,150,6


### Explicación
Filtro por condición usando `loc`: selecciona todas las filas donde `Ciudad == "Madrid"`. Mantiene estructura y permite encadenar más selecciones.


In [4]:
df.iloc[1:3]

,Ciudad,Ventas,Unidades
1,Sevilla,200,7
2,Madrid,150,6


### Explicación
Rebanado posicional con `iloc[1:3]`: devuelve filas 1 y 2 (límite superior exclusivo). Útil para submuestras rápidas sin depender de etiquetas.


### MultiIndex

In [5]:
df_mi = df.set_index(['Ciudad','Unidades'])
df_mi

,,Ventas
Ciudad,Unidades,
Madrid,5,100
Sevilla,7,200
Madrid,6,150
Valencia,4,130


### Explicación
Construimos un MultiIndex con `set_index(['Ciudad','Unidades'])` para permitir consultas jerárquicas y agregaciones más precisas sobre combinaciones de niveles.


In [6]:
df_mi.xs('Madrid', level='Ciudad')

,Ventas
Unidades,
5,100
6,150


### Explicación
`xs('Madrid', level='Ciudad')` extrae la sección del MultiIndex para la ciudad dada. `xs` facilita seleccionar por nivel sin tener que especificar todos los niveles superiores.


## 2. Filtros y expresiones avanzadas

In [7]:
df[df['Ventas'].between(120,180)]

,Ciudad,Ventas,Unidades
2,Madrid,150,6
3,Valencia,130,4


### Explicación
Uso de `between(120,180)` para crear un filtro inclusivo en ambos extremos sin escribir dos comparaciones. Devuelve filas con ventas en el rango.


In [8]:
df.query('Ciudad in ["Madrid","Sevilla"] and Ventas > 120')

,Ciudad,Ventas,Unidades
1,Sevilla,200,7
2,Madrid,150,6


### Explicación
`query` con `in [...]` y condición extra permite escribir filtros legibles sobre listas y operadores lógicos sin exceso de paréntesis.


In [9]:
df.eval('valor_medio = Ventas / Unidades')

,Ciudad,Ventas,Unidades,valor_medio
0,Madrid,100,5,20.000000
1,Sevilla,200,7,28.571429
2,Madrid,150,6,25.000000
3,Valencia,130,4,32.500000


### Explicación
`eval` crea una nueva columna calculada (`valor_medio`) de forma vectorizada y más concisa que asignar con expresión tradicional. Útil para expresiones largas.


## 3. GroupBy avanzado

In [10]:
df.groupby('Ciudad').agg(
    media_ventas=('Ventas','mean'),
    total_ventas=('Ventas','sum'),
    max_unidades=('Unidades','max')
)

,media_ventas,total_ventas,max_unidades
Ciudad,,,
Madrid,125.0,250,6
Sevilla,200.0,200,7
Valencia,130.0,130,4


### Explicación
Agregaciones nombradas con sintaxis de tuplas: cada clave del dict se convierte en nueva columna (`media_ventas`, `total_ventas`, etc.), facilitando resultados claros.


In [13]:
df.groupby('Ciudad').apply(lambda g: g[g['Ventas']==g['Ventas'].max()],include_groups=False)

,,Ventas,Unidades
Ciudad,,,
Madrid,2,150,6
Sevilla,1,200,7
Valencia,3,130,4


### Explicación
`apply` por grupo para extraer filas cuya `Ventas` es máxima en cada ciudad; devuelve DataFrame combinado preservando índice original.


## 4. Apply / Transform / Pipe con ejemplos complejos

In [15]:
def escala(x): return (x - x.min())/(x.max()-x.min())
df['Ventas_norm'] = df.groupby('Ciudad')['Ventas'].transform(escala)
df

,Ciudad,Ventas,Unidades,Ventas_norm
0,Madrid,100,5,0.0
1,Sevilla,200,7,NaN
2,Madrid,150,6,1.0
3,Valencia,130,4,NaN


### Explicación
`transform` con función de normalización por grupo crea columna comparable entre ciudades. Mantiene número de filas ideal para ingeniería de características.


In [17]:
(df.pipe(lambda d: d.assign(V2=d['Ventas']*2))
      .pipe(lambda d: d[d['V2']>250]))

,Ciudad,Ventas,Unidades,Ventas_norm,V2
1,Sevilla,200,7,NaN,400
2,Madrid,150,6,1.0,300
3,Valencia,130,4,NaN,260


### Explicación
Uso encadenado de `pipe` para mantener flujo funcional: primero crea columna derivada (`V2`), luego filtra. `pipe` facilita reordenar pasos y pruebas.


## 5. Ventanas móviles y temporales

In [18]:
s = pd.Series(range(1,21))
s.rolling(5).sum()

0      NaN
1      NaN
2      NaN
3      NaN
4     15.0
5     20.0
6     25.0
7     30.0
8     35.0
9     40.0
10    45.0
11    50.0
12    55.0
13    60.0
14    65.0
15    70.0
16    75.0
17    80.0
18    85.0
19    90.0
dtype: float64

### Explicación
Serie de 1..20 y suma móvil de ventana 5: suaviza fluctuaciones y prepara indicadores agregados temporales.


In [19]:
s.expanding().mean()

0      1.0
1      1.5
2      2.0
3      2.5
4      3.0
5      3.5
6      4.0
7      4.5
8      5.0
9      5.5
10     6.0
11     6.5
12     7.0
13     7.5
14     8.0
15     8.5
16     9.0
17     9.5
18    10.0
19    10.5
dtype: float64

### Explicación
`expanding().mean()` calcula media acumulada progresiva útil para analizar convergencia o tendencias generales.


In [20]:
s.ewm(alpha=0.2).mean()

0      1.000000
1      1.555556
2      2.147541
3      2.775068
4      3.436935
5      4.131668
6      4.857566
7      5.612752
8      6.395223
9      7.202902
10     8.033686
11     8.885484
12     9.756258
13    10.644052
14    11.547012
15    12.463404
16    13.391625
17    14.330208
18    15.277823
19    16.233274
dtype: float64

### Explicación
`ewm(alpha=0.2).mean()` calcula media exponencial ponderando más los valores recientes; útil para series temporales con reacción rápida.


## 6. Fechas y resampling avanzado

In [21]:
fechas = pd.date_range('2024-01-01', periods=10, freq='D')
df2 = pd.DataFrame({'Fecha':fechas, 'Ventas':np.random.randint(10,100,10)})
df2

,Fecha,Ventas
0,2024-01-01,30
1,2024-01-02,92
2,2024-01-03,54
3,2024-01-04,53
4,2024-01-05,58
5,2024-01-06,32
6,2024-01-07,35
7,2024-01-08,25
8,2024-01-09,20
9,2024-01-10,61


### Explicación
Generamos rango de fechas y DataFrame temporal para practicar resampling y agregaciones sobre períodos homogéneos.


In [22]:
df2.resample('3D', on='Fecha').sum()

,Ventas
Fecha,
2024-01-01,176
2024-01-04,143
2024-01-07,80
2024-01-10,61


### Explicación
`resample('3D', on='Fecha').sum()` agrupa cada bloque de 3 días tomando la suma de ventas; útil para reducir granularidad manteniendo totales.


## 7. Joins avanzados

In [23]:
dfA = pd.DataFrame({'id':[1,2,3,4], 'valor':[10,20,30,40]})
dfB = pd.DataFrame({'id':[3,4,5], 'cat':['A','B','C']})
dfA.merge(dfB, on='id', how='outer')

,id,valor,cat
0,1,10.0,NaN
1,2,20.0,NaN
2,3,30.0,A
3,4,40.0,B
4,5,NaN,C


### Explicación
Ejemplo de `merge` externo con columnas diferentes para observar cómo se introducen NaN donde faltan coincidencias (`outer`).


In [24]:
dfA.merge(dfB, on='id', how='inner')

,id,valor,cat
0,3,30,A
1,4,40,B


### Explicación
`inner` merge conserva sólo ids presentes en ambos DataFrames; útil para intersecciones de claves.


## 8. Merge con condiciones

In [25]:
dfL = pd.DataFrame({'x':[1,5,9]})
dfR = pd.DataFrame({'y':[2,4,6,8]})

(dfL.merge(dfR, how='cross')
    .query('x < y'))

,x,y
0,1,2
1,1,4
2,1,6
3,1,8
6,5,6
7,5,8


### Explicación
`merge` con `how='cross'` genera producto cartesiano y luego se filtra por condición (`x < y`). Útil para explorar combinaciones posibles.


## 9. Intervalos e interval joins

In [26]:
intervals = pd.IntervalIndex.from_breaks([0,10,20,30])
df_int = pd.DataFrame({'intervalo': intervals, 'categoria':['baja','media','alta']})
df_int

,intervalo,categoria
0,"(0, 10]",baja
1,"(10, 20]",media
2,"(20, 30]",alta


### Explicación
`IntervalIndex.from_breaks` crea intervalos contiguos; se usa para categorizar valores por rangos y luego mapear a etiquetas descriptivas.


In [27]:
df_vals = pd.DataFrame({'valor':[3,7,12,25]})
pd.merge_asof(df_vals.sort_values('valor'),
              df_int.assign(start=df_int['intervalo'].apply(lambda x: x.left),
                            end=df_int['intervalo'].apply(lambda x: x.right)
                           ).sort_values('start'),
              left_on='valor', right_on='start')

,valor,intervalo,categoria,start,end
0,3,"(0, 10]",baja,0,10
1,7,"(0, 10]",baja,0,10
2,12,"(10, 20]",media,10,20
3,25,"(20, 30]",alta,20,30


### Explicación
`merge_asof` realiza unión aproximada ordenada (por cercanía hacia atrás). Aquí se prepara DataFrame de intervalos con límites para asociar cada valor al inicio de intervalo más próximo.


## 10. Optimización

In [28]:
df_opt = df.copy()
df_opt['Ciudad'] = df_opt['Ciudad'].astype('category')
df_opt.dtypes

Ciudad         category
Ventas            int64
Unidades          int64
Ventas_norm     float64
dtype: object

### Explicación
Conversión a `category` reduce memoria y acelera operaciones basadas en repetición de valores (comparaciones, groupby), útil en columnas con pocos valores distintos.
